In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/badcodebuilder/insdn-dataset/InSDN_DatasetCSV/metasploitable-2.csv
/kaggle/input/datasets/badcodebuilder/insdn-dataset/InSDN_DatasetCSV/Normal_data.csv
/kaggle/input/datasets/badcodebuilder/insdn-dataset/InSDN_DatasetCSV/OVS.csv
/kaggle/input/datasets/dhoogla/csecicids2018/DoS2-Friday-16-02-2018_TrafficForML_CICFlowMeter.parquet
/kaggle/input/datasets/dhoogla/csecicids2018/DoS1-Thursday-15-02-2018_TrafficForML_CICFlowMeter.parquet
/kaggle/input/datasets/dhoogla/csecicids2018/DDoS1-Tuesday-20-02-2018_TrafficForML_CICFlowMeter.parquet
/kaggle/input/datasets/dhoogla/csecicids2018/Web2-Friday-23-02-2018_TrafficForML_CICFlowMeter.parquet
/kaggle/input/datasets/dhoogla/csecicids2018/Web1-Thursday-22-02-2018_TrafficForML_CICFlowMeter.parquet
/kaggle/input/datasets/dhoogla/csecicids2018/Botnet-Friday-02-03-2018_TrafficForML_CICFlowMeter.parquet
/kaggle/input/datasets/dhoogla/csecicids2018/Infil2-Thursday-01-03-2018_TrafficForML_CICFlowMeter.parquet
/kaggle/input/datasets

In [5]:
!rm -rf TG_GAT_DDoS_Detection   # أو اسم المجلد القديم إذا كان مختلفًا


In [6]:
!git clone https://github.com/hosamalhmiry738-dotcom/TG-GAT-DDoS-Detection.git TG_GAT_DDoS_Detection


fatal: destination path 'TG-GAT-DDoS-Detection' already exists and is not an empty directory.
/kaggle/working/TG-GAT-DDoS-Detection
^C
ERROR: Operation cancelled by user


In [3]:
import os

# إعادة تسمية المجلد من "TG-GAT-DDoS-Detection" إلى "TG_GAT_DDoS_Detection"
!mv TG-GAT-DDoS-Detection TG_GAT_DDoS_Detection
%cd TG_GAT_DDoS_Detection
!pip install -r requirements.txt

fatal: destination path 'TG-GAT-DDoS-Detection' already exists and is not an empty directory.


# TG-GAT DDoS Detection Training on Kaggle

This notebook implements the complete training pipeline for the TG-GAT (Transformer-Gated Graph Attention Network) model for DDoS detection as described in the research paper.

## Features:
- **Multi-dataset integration**: CIC-DDoS2019, CSE-CIC-IDS2018, InSDN
- **Dynamic graph construction**: 100ms temporal windows
- **Hybrid architecture**: GAT + Transformer + GRU
- **Zero-Day attack generation**: GAN-based synthetic attacks
- **Real-time performance**: <20ms detection time
- **Explainable AI**: GNNExplainer for attack interpretation
- **W&B integration**: Comprehensive experiment tracking

## 1. Setup and Installation

In [ ]:
# Install required packages
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install torch-geometric torch-scatter torch-sparse torch-cluster torch-spline-conv -f https://data.pyg.org/whl/torch-1.13.0+cu118.html
!pip install transformers wandb pyyaml omegaconf
!pip install pandas numpy scikit-learn matplotlib seaborn plotly
!pip install pyarrow fastparquet openpyxl
!pip install captum shap lime
!pip install psutil GPUtil

print("Packages installed successfully!")

In [ ]:
# Import libraries
import os
import sys
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import wandb
import yaml
from pathlib import Path
import warnings
from tqdm import tqdm
import time
import json

# Add src to path
sys.path.append('/kaggle/working/TG_GAT_DDoS_Detection/src')

# Set up warnings
warnings.filterwarnings('ignore')

# Check GPU availability
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

print("\nLibraries imported successfully!")

## 2. Clone Project Repository

In [ ]:
# Clone the TG-GAT repository (replace with your actual repository URL)
!git clone  https://github.com/hosamalhmiry738-dotcom/TG_GAT_DDoS_Detection.git

# Navigate to project directory
%cd TG_GAT_DDoS_Detection

# Verify project structure
!ls -la

print("Repository cloned successfully!")

## 3. Load and Configure

In [ ]:
# Load configuration
from src.utils.config_loader import ConfigLoader
from src.utils.wandb_logger import WandbLogger

# Load configuration
config_loader = ConfigLoader('config/default.yaml')
config = config_loader.get_config()

# Update configuration for Kaggle environment
config['hardware']['device'] = 'cuda' if torch.cuda.is_available() else 'cpu'
config['training']['batch_size'] = 16 if torch.cuda.is_available() else 8  # Adjust based on GPU memory
config['training']['epochs'] = 10  # Can be increased for better results

# Create necessary directories
config_loader.create_directories()

# Print configuration
print("Configuration loaded successfully!")
print(f"\nModel Configuration:")
model_config = config['model']
for key, value in model_config.items():
    print(f"  {key}: {value}")

print(f"\nTraining Configuration:")
training_config = config['training']
for key, value in training_config.items():
    print(f"  {key}: {value}")

## 4. Initialize W&B Logging

In [ ]:
# Initialize W&B
# Set your W&B API key here or use Kaggle secrets
# wandb.login(key='your-wandb-api-key')

# Initialize logger
wandb_logger = WandbLogger(
    config=config,
    project_name="TG_GAT_DDoS_Detection",
    run_name=f"tg-gat-kaggle-{int(time.time())}",
    tags=['ddos-detection', 'tg-gat', 'kaggle']
)

wandb_logger.init_run()

print("W&B logging initialized!")

## 5. Load and Preprocess Datasets

In [ ]:
# Import data processing modules
from src.data.preprocessing import DataPreprocessor
from src.data.graph_builder import GraphBuilder
from src.data.gan_generator import GANGenerator

# Initialize data preprocessor
preprocessor = DataPreprocessor(config)

# Define dataset paths (Kaggle datasets)
dataset_paths = {
    'CIC-DDoS2019': [
        '/kaggle/input/cic-ddos2019/CSV-01-12/01-12/UDP-training.parquet',
        '/kaggle/input/cic-ddos2019/CSV-01-12/01-12/UDP-testing.parquet',
        '/kaggle/input/cic-ddos2019/CSV-01-12/01-12/SYN-training.parquet',
        '/kaggle/input/cic-ddos2019/CSV-01-12/01-12/SYN-testing.parquet'
    ],
    'CSE-CIC-IDS2018': [
        '/kaggle/input/cse-cic-ids2018/CSV-02-14/02-14/Botnet-Friday-02-03-2018_TrafficForML_CICFlowMeter.parquet',
        '/kaggle/input/cse-cic-ids2018/CSV-02-14/02-14/DDoS1-Tuesday-20-02-2018_TrafficForML_CICFlowMeter.parquet',
        '/kaggle/input/cse-cic-ids2018/CSV-02-14/02-14/DDoS2-Wednesday-21-02-2018_TrafficForML_CICFlowMeter.parquet'
    ],
    'InSDN': [
        '/kaggle/input/insdn-dataset/Normal_data.csv',
        '/kaggle/input/insdn-dataset/OVS.csv',
        '/kaggle/input/insdn-dataset/metasploitable-2.csv'
    ]
}

# Check which datasets are available
available_datasets = {}
for dataset_type, paths in dataset_paths.items():
    available_paths = []
    for path in paths:
        if os.path.exists(path):
            available_paths.append(path)
    if available_paths:
        available_datasets[dataset_type] = available_paths

print(f"Available datasets: {list(available_datasets.keys())}")
for dataset_type, paths in available_datasets.items():
    print(f"  {dataset_type}: {len(paths)} files")

In [ ]:
# Process datasets
if available_datasets:
    print("Processing datasets...")
    
    # Process and combine datasets
    combined_data = preprocessor.process_multiple_datasets(available_datasets)
    
    print(f"Combined dataset shape: {combined_data.shape}")
    print(f"Class distribution:")
    print(combined_data['Label'].value_counts())
    
    # Split data
    train_df, val_df, test_df = preprocessor.split_data(combined_data)
    
    print(f"\nData splits:")
    print(f"  Train: {len(train_df)} samples")
    print(f"  Validation: {len(val_df)} samples")
    print(f"  Test: {len(test_df)} samples")
    
    # Get feature columns
    feature_columns = preprocessor.get_feature_columns(combined_data)
    print(f"\nNumber of features: {len(feature_columns)}")
else:
    print("No datasets available. Please add datasets to Kaggle.")
    # Create dummy data for testing
    print("Creating dummy data for testing...")
    num_samples = 1000
    num_features = 80
    
    # Create dummy DataFrame
    dummy_data = pd.DataFrame(
        np.random.randn(num_samples, num_features),
        columns=[f'feature_{i}' for i in range(num_features)]
    )
    dummy_data['Label'] = np.random.choice(['Benign', 'DDoS'], num_samples)
    
    # Process dummy data
    dummy_data = preprocessor.preprocess_dataset(dummy_data, 'CIC-DDoS2019')
    dummy_data = preprocessor.normalize_features(dummy_data, fit_scaler=True)
    
    # Split dummy data
    train_df, val_df, test_df = preprocessor.split_data(dummy_data)
    
    feature_columns = preprocessor.get_feature_columns(dummy_data)
    
    print(f"Dummy dataset created: {dummy_data.shape}")

## 6. Build Dynamic Graphs

In [ ]:
# Initialize graph builder
graph_builder = GraphBuilder(config)

# Build dynamic graphs from training data
print("Building dynamic graphs from training data...")
train_graphs = graph_builder.build_dynamic_graphs(train_df)

print(f"Built {len(train_graphs)} training graphs")
if train_graphs:
    print(f"Sample graph stats:")
    sample_graph = train_graphs[0]
    print(f"  Nodes: {sample_graph.num_nodes}")
    print(f"  Edges: {sample_graph.num_edges}")
    print(f"  Features: {sample_graph.x.shape if hasattr(sample_graph, 'x') else 'N/A'}")

# Build graphs from validation data
print("\nBuilding dynamic graphs from validation data...")
val_graphs = graph_builder.build_dynamic_graphs(val_df)
print(f"Built {len(val_graphs)} validation graphs")

# Build graphs from test data
print("\nBuilding dynamic graphs from test data...")
test_graphs = graph_builder.build_dynamic_graphs(test_df)
print(f"Built {len(test_graphs)} test graphs")

## 7. Generate Zero-Day Attacks with GAN

In [ ]:
# Initialize GAN generator
gan_generator = GANGenerator(config)

# Prepare training data for GAN (only DDoS samples)
ddos_samples = train_df[train_df['Label'] == 'DDoS']
if len(ddos_samples) > 0:
    # Convert to tensor
    ddos_features = ddos_samples[feature_columns].values
    ddos_tensor = torch.FloatTensor(ddos_features)
    
    print(f"Training GAN on {len(ddos_samples)} DDoS samples...")
    
    # Train GAN
    gan_generator.train(ddos_tensor, epochs=50, batch_size=32)
    
    # Generate synthetic Zero-Day attacks
    num_synthetic = min(1000, len(ddos_samples) // 2)
    synthetic_attacks = gan_generator.generate_samples(num_synthetic)
    
    print(f"Generated {num_synthetic} synthetic Zero-Day attacks")
    
    # Evaluate quality of generated attacks
    quality_metrics = gan_generator.evaluate_quality(ddos_tensor, synthetic_attacks)
    print(f"GAN quality metrics: {quality_metrics}")
    
    # Log to W&B
    wandb_logger.log_metrics(quality_metrics, step=0)
else:
    print("No DDoS samples found for GAN training")
    synthetic_attacks = None

## 8. Initialize TG-GAT Model

In [ ]:
# Import TG-GAT model
from src.models.tg_gat import TGGATModel

# Initialize model
print("Initializing TG-GAT model...")
model = TGGATModel(config).to(device)

# Print model information
model_info = model.get_model_info()
print(f"Model information:")
for key, value in model_info.items():
    print(f"  {key}: {value}")

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTotal parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

# Log model to W&B
wandb_logger.log_metrics({'total_parameters': total_params, 'trainable_parameters': trainable_params}, step=0)

## 9. Create Data Loaders

In [ ]:
from torch.utils.data import DataLoader, TensorDataset
from torch_geometric.data import Batch

# Create labels for graphs
def create_graph_labels(graphs, labels_df):
    """Create labels for graphs based on the majority label in each time window."""
    graph_labels = []
    
    for i, graph in enumerate(graphs):
        # For simplicity, assign labels based on index (in practice, this should be based on actual timestamps)
        if i < len(labels_df):
            label = 1 if labels_df.iloc[i]['Label'] == 'DDoS' else 0
        else:
            label = 0  # Default to Benign
        graph_labels.append(label)
    
    return torch.tensor(graph_labels, dtype=torch.long)

# Create graph datasets
train_labels = create_graph_labels(train_graphs, train_df)
val_labels = create_graph_labels(val_graphs, val_df)
test_labels = create_graph_labels(test_graphs, test_df)

print(f"Training labels distribution: {torch.bincount(train_labels)}")
print(f"Validation labels distribution: {torch.bincount(val_labels)}")
print(f"Test labels distribution: {torch.bincount(test_labels)}")

# Create custom collate function
def collate_fn(graphs):
    """Custom collate function for graph batches."""
    return Batch.from_data_list(graphs)

# Create data loaders
batch_size = config['training']['batch_size']

train_loader = DataLoader(
    train_graphs, 
    batch_size=batch_size, 
    shuffle=True, 
    collate_fn=collate_fn,
    num_workers=2,
    pin_memory=True if torch.cuda.is_available() else False
)

val_loader = DataLoader(
    val_graphs, 
    batch_size=batch_size, 
    shuffle=False, 
    collate_fn=collate_fn,
    num_workers=2,
    pin_memory=True if torch.cuda.is_available() else False
)

test_loader = DataLoader(
    test_graphs, 
    batch_size=batch_size, 
    shuffle=False, 
    collate_fn=collate_fn,
    num_workers=2,
    pin_memory=True if torch.cuda.is_available() else False
)

print(f"\nData loaders created:")
print(f"  Train batches: {len(train_loader)}")
print(f"  Validation batches: {len(val_loader)}")
print(f"  Test batches: {len(test_loader)}")

## 10. Initialize Trainer

In [ ]:
# Import trainer
from src.training.trainer import Trainer

# Initialize trainer
print("Initializing trainer...")
trainer = Trainer(config, model)

print("Trainer initialized successfully!")
print(f"Device: {trainer.device}")
print(f"Batch size: {trainer.batch_size}")
print(f"Learning rate: {trainer.learning_rate}")
print(f"Epochs: {trainer.epochs}")

## 11. Train the Model

In [ ]:
# Start training
print("Starting TG-GAT model training...")
print(f"Training for {trainer.epochs} epochs")
print(f"Using device: {trainer.device}")

# Track training time
start_time = time.time()

# Train the model
training_results = trainer.train(train_loader, val_loader)

training_time = time.time() - start_time

print(f"\nTraining completed in {training_time:.2f} seconds")
print(f"Best validation F1: {trainer.best_val_f1:.4f}")

# Log training time to W&B
wandb_logger.log_metrics({'training_time_seconds': training_time}, step=trainer.epochs)

# Save training results
results_path = 'outputs/training_results.json'
with open(results_path, 'w') as f:
    json.dump(training_results, f, indent=2, default=str)

print(f"Training results saved to {results_path}")

## 12. Evaluate the Model

In [ ]:
# Import evaluator
from src.evaluation.test import ModelTester

# Initialize evaluator
print("Initializing model evaluator...")
evaluator = ModelTester(config, 'checkpoints/best_model.pth')

# Run comprehensive evaluation
print("Running comprehensive model evaluation...")
evaluation_results = evaluator.run_full_evaluation(test_loader)

# Extract key metrics
if 'standard' in evaluation_results:
    test_metrics = evaluation_results['standard']['metrics']
    print(f"\nTest Results:")
    print(f"  Accuracy: {test_metrics.get('accuracy', 0):.4f}")
    print(f"  F1-Score: {test_metrics.get('f1', 0):.4f}")
    print(f"  Precision: {test_metrics.get('precision', 0):.4f}")
    print(f"  Recall: {test_metrics.get('recall', 0):.4f}")
    print(f"  False Positive Rate: {test_metrics.get('fpr', 0):.4f}")
    print(f"  ROC AUC: {test_metrics.get('roc_auc', 0):.4f}")
    print(f"  Detection Time: {test_metrics.get('avg_detection_time', 0):.4f}s")
    
    # Log test metrics to W&B
    wandb_logger.log_test_metrics(test_metrics)

# Save evaluation results
eval_results_path = 'outputs/evaluation_results.json'
with open(eval_results_path, 'w') as f:
    json.dump(evaluation_results, f, indent=2, default=str)

print(f"\nEvaluation results saved to {eval_results_path}")

## 13. Generate Explanations with XAI

In [ ]:
# Import XAI modules
from src.evaluation.xai import XAIExplainer

# Initialize XAI explainer
print("Initializing XAI explainer...")
xai_explainer = XAIExplainer(model, config)

# Generate explanations for a few test samples
print("Generating explanations for test samples...")

explanations = []
num_explanations = min(5, len(test_graphs))

for i in range(num_explanations):
    try:
        # Get a test graph
        test_graph = test_graphs[i]
        
        # Generate explanation
        explanation = xai_explainer.explain_prediction(
            test_graph, 
            target_class=1,  # DDoS class
            visualize=True
        )
        
        # Generate human-readable report
        report = xai_explainer.generate_explanation_report(explanation)
        
        explanations.append({
            'graph_index': i,
            'explanation': explanation,
            'report': report
        })
        
        print(f"Generated explanation for graph {i+1}")
        
    except Exception as e:
        print(f"Error generating explanation for graph {i}: {str(e)}")
        continue

print(f"\nGenerated {len(explanations)} explanations")

# Save explanations
if explanations:
    explanations_path = 'outputs/explanations.json'
    # Convert explanations to JSON-serializable format
    serializable_explanations = []
    for exp in explanations:
        serializable_exp = {
            'graph_index': exp['graph_index'],
            'report': exp['report']
        }
        serializable_explanations.append(serializable_exp)
    
    with open(explanations_path, 'w') as f:
        json.dump(serializable_explanations, f, indent=2)
    
    print(f"Explanations saved to {explanations_path}")
    
    # Print a sample explanation
    print("\nSample Explanation:")
    print("=" * 50)
    print(explanations[0]['report'])

## 14. Visualize Results

In [ ]:
# Create visualization plots
import matplotlib.pyplot as plt
import seaborn as sns

# Set up plotting style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

# Create performance comparison plot
if 'standard' in evaluation_results:
    test_metrics = evaluation_results['standard']['metrics']
    
    # Create performance metrics plot
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    
    # Accuracy and F1
    metrics = ['Accuracy', 'F1-Score', 'Precision', 'Recall']
    values = [
        test_metrics.get('accuracy', 0),
        test_metrics.get('f1', 0),
        test_metrics.get('precision', 0),
        test_metrics.get('recall', 0)
    ]
    
    axes[0, 0].bar(metrics, values, color=['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728'])
    axes[0, 0].set_title('Classification Metrics')
    axes[0, 0].set_ylabel('Score')
    axes[0, 0].set_ylim(0, 1)
    
    # Add value labels on bars
    for i, v in enumerate(values):
        axes[0, 0].text(i, v + 0.01, f'{v:.3f}', ha='center', va='bottom')
    
    # ROC AUC
    roc_auc = test_metrics.get('roc_auc', 0)
    axes[0, 1].bar(['ROC AUC'], [roc_auc], color='#9467bd')
    axes[0, 1].set_title('ROC AUC')
    axes[0, 1].set_ylabel('Score')
    axes[0, 1].set_ylim(0, 1)
    axes[0, 1].text(0, roc_auc + 0.01, f'{roc_auc:.3f}', ha='center', va='bottom')
    
    # False Positive Rate
    fpr = test_metrics.get('fpr', 0)
    axes[1, 0].bar(['False Positive Rate'], [fpr], color='#8c564b')
    axes[1, 0].set_title('False Positive Rate')
    axes[1, 0].set_ylabel('Rate')
    axes[1, 0].text(0, fpr + 0.001, f'{fpr:.4f}', ha='center', va='bottom')
    
    # Detection Time
    detection_time = test_metrics.get('avg_detection_time', 0)
    axes[1, 1].bar(['Detection Time (s)'], [detection_time], color='#e377c2')
    axes[1, 1].set_title('Average Detection Time')
    axes[1, 1].set_ylabel('Time (seconds)')
    axes[1, 1].text(0, detection_time + 0.001, f'{detection_time:.4f}', ha='center', va='bottom')
    
    plt.tight_layout()
    plt.savefig('outputs/performance_metrics.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    # Log plot to W&B
    wandb_logger.log_custom_plot('performance_metrics', fig)

# Create training curves plot
if 'training_history' in training_results:
    history = training_results['training_history']
    
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    
    # Training and validation loss
    epochs = range(1, len(history['train_loss']) + 1)
    axes[0, 0].plot(epochs, history['train_loss'], label='Train Loss', color='blue')
    if history['val_loss']:
        axes[0, 0].plot(epochs, history['val_loss'], label='Val Loss', color='red')
    axes[0, 0].set_title('Training and Validation Loss')
    axes[0, 0].set_xlabel('Epoch')
    axes[0, 0].set_ylabel('Loss')
    axes[0, 0].legend()
    axes[0, 0].grid(True)
    
    # F1 Score
    train_f1 = [m.get('f1', 0) for m in history['train_metrics']]
    val_f1 = [m.get('f1', 0) for m in history['val_metrics']]
    
    axes[0, 1].plot(epochs, train_f1, label='Train F1', color='blue')
    if val_f1:
        axes[0, 1].plot(epochs, val_f1, label='Val F1', color='red')
    axes[0, 1].set_title('F1 Score')
    axes[0, 1].set_xlabel('Epoch')
    axes[0, 1].set_ylabel('F1 Score')
    axes[0, 1].legend()
    axes[0, 1].grid(True)
    
    # Accuracy
    train_acc = [m.get('accuracy', 0) for m in history['train_metrics']]
    val_acc = [m.get('accuracy', 0) for m in history['val_metrics']]
    
    axes[1, 0].plot(epochs, train_acc, label='Train Accuracy', color='blue')
    if val_acc:
        axes[1, 0].plot(epochs, val_acc, label='Val Accuracy', color='red')
    axes[1, 0].set_title('Accuracy')
    axes[1, 0].set_xlabel('Epoch')
    axes[1, 0].set_ylabel('Accuracy')
    axes[1, 0].legend()
    axes[1, 0].grid(True)
    
    # Learning rate
    if history['learning_rates']:
        axes[1, 1].plot(epochs, history['learning_rates'], color='green')
        axes[1, 1].set_title('Learning Rate')
        axes[1, 1].set_xlabel('Epoch')
        axes[1, 1].set_ylabel('Learning Rate')
        axes[1, 1].grid(True)
    
    plt.tight_layout()
    plt.savefig('outputs/training_curves.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    # Log plot to W&B
    wandb_logger.log_custom_plot('training_curves', fig)

print("Visualizations saved to outputs/")

## 15. Final Summary and Cleanup

In [ ]:
# Generate final summary
print("=" * 60)
print("TG-GAT DDoS Detection Training Summary")
print("=" * 60)

# Model information
print(f"\nModel Configuration:")
print(f"  Architecture: TG-GAT (Transformer + GAT + GRU)")
print(f"  Hidden Dimension: {model_info['hidden_dim']}")
print(f"  Attention Heads: {model_info['num_heads']}")
print(f"  Total Parameters: {model_info['total_parameters']:,}")

# Training results
print(f"\nTraining Results:")
print(f"  Epochs Completed: {trainer.epochs}")
print(f"  Training Time: {training_time:.2f} seconds")
print(f"  Best Validation F1: {trainer.best_val_f1:.4f}")

# Test results
if 'standard' in evaluation_results:
    test_metrics = evaluation_results['standard']['metrics']
    print(f"\nTest Results:")
    print(f"  Accuracy: {test_metrics.get('accuracy', 0):.4f}")
    print(f"  F1-Score: {test_metrics.get('f1', 0):.4f}")
    print(f"  Precision: {test_metrics.get('precision', 0):.4f}")
    print(f"  Recall: {test_metrics.get('recall', 0):.4f}")
    print(f"  False Positive Rate: {test_metrics.get('fpr', 0):.4f}")
    print(f"  ROC AUC: {test_metrics.get('roc_auc', 0):.4f}")
    print(f"  Detection Time: {test_metrics.get('avg_detection_time', 0):.4f}s")

# Performance targets achievement
print(f"\nPerformance Targets Achievement:")
targets = {
    'accuracy': 0.998,
    'f1': 0.997,
    'fpr': 0.005,
    'detection_time': 0.020
}

for metric, target in targets.items():
    if metric in test_metrics:
        value = test_metrics[metric]
        if metric == 'fpr':
            achieved = value <= target
            status = "✓" if achieved else "✗"
        elif metric == 'detection_time':
            achieved = value <= target
            status = "✓" if achieved else "✗"
        else:
            achieved = value >= target
            status = "✓" if achieved else "✗"
        
        print(f"  {metric}: {value:.4f} (target: {target:.4f}) {status}")

# Overall achievement
if 'standard' in evaluation_results:
    overall_achievement = test_metrics.get('overall_achievement', 0)
    print(f"\nOverall Achievement: {overall_achievement:.4f} ({overall_achievement*100:.1f}%)")

# Files created
print(f"\nGenerated Files:")
output_files = [
    'outputs/training_results.json',
    'outputs/evaluation_results.json',
    'outputs/explanations.json',
    'outputs/performance_metrics.png',
    'outputs/training_curves.png'
]

for file_path in output_files:
    if os.path.exists(file_path):
        print(f"  ✓ {file_path}")
    else:
        print(f"  ✗ {file_path} (not found)")

print("\n" + "=" * 60)
print("Training completed successfully!")
print("=" * 60)

In [ ]:
# Finish W&B run
wandb_logger.finish_run()

print("W&B run finished. Check your dashboard for detailed results!")

## 16. Save Model for Deployment

If you want to save the trained model for deployment, you can download the checkpoint files from the Kaggle output directory.

In [ ]:
# Create deployment package
import shutil

# Create deployment directory
deployment_dir = 'deployment_package'
os.makedirs(deployment_dir, exist_ok=True)

# Copy necessary files
files_to_copy = [
    'checkpoints/best_model.pth',
    'config/default.yaml',
    'src/',
    'requirements.txt',
    'README.md'
]

for file_path in files_to_copy:
    if os.path.exists(file_path):
        if os.path.isfile(file_path):
            shutil.copy2(file_path, deployment_dir)
        else:
            shutil.copytree(file_path, os.path.join(deployment_dir, os.path.basename(file_path)))
        print(f"Copied {file_path}")
    else:
        print(f"File not found: {file_path}")

# Create deployment script
deployment_script = '''
#!/bin/bash
# TG-GAT DDoS Detection Deployment Script

echo "Installing dependencies..."
pip install -r requirements.txt

echo "TG-GAT DDoS Detection model is ready for deployment!"
echo "Use the src/ module to integrate with your applications."
'''

with open(os.path.join(deployment_dir, 'deploy.sh'), 'w') as f:
    f.write(deployment_script)

print(f"\nDeployment package created in {deployment_dir}/")
print("You can download this directory for deployment.")